In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/ev-dataset-model/model_data.csv
/kaggle/input/datasets/sahilhussain2410/ev-predction/ev_predictions/demand_predictions_valid.csv
/kaggle/input/datasets/sahilhussain2410/ev-predction/ev_predictions/demand_predictions.csv


In [2]:

model_data = pd.read_csv(
    "/kaggle/input/datasets/sahilhussain2410/ev-dataset-model/model_data.csv",
    parse_dates=["hour"]
)

# Demand model predictions
pred_df = pd.read_csv(
    "/kaggle/input/datasets/sahilhussain2410/ev-predction/ev_predictions/demand_predictions.csv",
    index_col=0
)

# Add predictions back
test_df = model_data.iloc[-len(pred_df):].copy()



In [3]:
test_df["predicted_demand"] =pred_df['predicted_demand']

In [4]:
test_df["predicted_utilization"] = (
    test_df["util_lag_24"]
)

In [5]:
BASE_PRICE = 15.0

LOW_UTIL = 0.30

HIGH_UTIL = 0.60

MAX_PRICE = 30.0

MIN_PRICE = 8.0

ALPHA = 0.50

BETA = 0.50

In [6]:
def dynamic_tariff(u):

    if u > HIGH_UTIL:

        price = (
            BASE_PRICE *
            (
                1 +
                ALPHA *
                (u - HIGH_UTIL)
            )
        )

    elif u < LOW_UTIL:

        price = (
            BASE_PRICE *
            (
                1 -
                BETA *
                (LOW_UTIL - u)
            )
        )

    else:

        price = BASE_PRICE

    return np.clip(
        price,
        MIN_PRICE,
        MAX_PRICE
    )

In [7]:
test_df["dynamic_price"] = (
    test_df["predicted_utilization"]
    .apply(dynamic_tariff)
)

In [8]:
ELASTICITY = -0.5
test_df["price_ratio"] = (
    test_df["dynamic_price"]
    /
    BASE_PRICE
)

In [9]:
test_df["adjusted_demand"] = (
    test_df["predicted_demand"]
    *
    (
        1 +
        ELASTICITY *
        (
            test_df["price_ratio"] - 1
        )
    )
)

In [10]:
test_df["adjusted_demand"] = (
    test_df["adjusted_demand"]
    .clip(lower=0)
)

In [11]:
test_df["baseline_revenue"] = (
    test_df["predicted_demand"]
    *
    BASE_PRICE
)

In [12]:
test_df["dynamic_revenue"] = (
    test_df["adjusted_demand"]
    *
    test_df["dynamic_price"]
)

In [13]:
test_df["new_utilization"] = (
    test_df["predicted_utilization"]
    *
    (
        test_df["adjusted_demand"]
        /
        test_df["predicted_demand"]
    )
)

In [14]:
test_df["new_utilization"] = (
    test_df["new_utilization"]
    .clip(0,1)
)

In [15]:
baseline_rev = (
    test_df["baseline_revenue"]
    .sum()
)

dynamic_rev = (
    test_df["dynamic_revenue"]
    .sum()
)

In [16]:
revenue_gain_pct = (
    (
        dynamic_rev -
        baseline_rev
    )
    /
    baseline_rev
) * 100
print(
    f"Revenue Gain %: {revenue_gain_pct:.2f}"
)

Revenue Gain %: 0.44


In [17]:
off_peak_mask = (
    test_df["predicted_utilization"]
    < 0.30
)
baseline_offpeak = (
    test_df.loc[
        off_peak_mask,
        "predicted_demand"
    ]
    .sum()
)
new_offpeak = (
    test_df.loc[
        off_peak_mask,
        "adjusted_demand"
    ]
    .sum()
)
offpeak_uplift = (
    (
        new_offpeak -
        baseline_offpeak
    )
    /
    baseline_offpeak
) * 100

In [18]:
print(
    f"Off Peak Uplift %: {offpeak_uplift:.2f}"
)

Off Peak Uplift %: 2.17


In [19]:
baseline_util = (
    test_df["predicted_utilization"]
    .mean()
)

new_util = (
    test_df["new_utilization"]
    .mean()
)

In [20]:
util_change_pct = (
    (
        new_util -
        baseline_util
    )
    /
    baseline_util
) * 100
print(
    f"Utilization Change %: {util_change_pct:.2f}"
)

Utilization Change %: 3.58


In [21]:
tariffs = test_df[
    [
        "entity_id",
        "hour",
        "dynamic_price"
    ]
]
tariffs.columns = [
    "station",
    "time",
    "new_price"
]
tariffs.to_csv(
    "tariffs.csv",
    index=False
)

In [22]:
pricing_report = pd.DataFrame({

    "Metric":[
        "Baseline Revenue",
        "Dynamic Revenue",
        "Revenue Gain %",
        "Off Peak Uplift %",
        "Utilization Change %"
    ],

    "Value":[
        baseline_rev,
        dynamic_rev,
        revenue_gain_pct,
        offpeak_uplift,
        util_change_pct
    ]
})
pricing_report.to_csv(
    "pricing_report.csv",
    index=False
)

In [23]:
test_df.shape

(25787, 38)

In [24]:
test_df.to_csv("test_df.csv",index=False)